<a href="https://colab.research.google.com/github/sangjkim930/AI-Driven-Research-Methodology/blob/main/01_API_Basics_and_Semantic_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI API Basics and Semantic Retrieval

This notebook introduces the basic building blocks used later in our RAG exercises.

### Workflow

**API Setup → Simple Request → Model Parameters → Embeddings → Cosine Similarity → Semantic Retrieval**

### Learning Objectives

In this exercise, you will learn how to:

1. connect Google Colab to the OpenAI API;
2. send a simple request to a language model;
3. adjust model parameters;
4. convert text into embeddings;
5. calculate semantic similarity; and
6. retrieve the most relevant document.

> **Important:** Run the cells in order from top to bottom.

### Step 0. Install the OpenAI Library

Install the OpenAI Python library required to connect Google Colab to the OpenAI API.

In [ ]:
!pip install -q openai

### Step 1. Connect to the OpenAI API

Retrieve your API key from Google Colab Secrets and create an OpenAI client.

Make sure your API key is stored under the name `OPENAI_API_KEY`.

In [1]:
import os

from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

> **Security Reminder:** Never write your API key directly in a notebook that will be shared publicly.

## Exercise 1. Send Your First API Request

We will begin with a simple request to the OpenAI API.

The prompt asks the model to briefly explain how generative AI can support academic research.

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Explain in three sentences how generative AI can support academic research."
)

print(response.output_text)

### Try It Yourself

Replace the text in `input` with a question from your own research field and run the cell again.

## Exercise 2. Control the Model Response

The API allows us to configure how the model approaches and presents a task.

In this exercise, we will use a more complex academic question and adjust the model settings.

In [ ]:
response = client.responses.create(
    model="gpt-5.6",
    input="Compare stakeholder theory and resource-based theory in environmental performance research.",
    reasoning={"effort": "high"},
    text={"verbosity": "low"}
)

print(response.output_text)

### Try It Yourself

Change the research question or one of the model settings and compare the response.

> **Research Tip:** Model names and supported parameters may change over time. Check the current OpenAI Platform documentation before using API code in a research workflow.

### From Generation to Retrieval

So far, we have asked a language model to generate answers.

But research workflows often require another important task:

**finding the most relevant evidence before generating an answer.**

We will now explore how embeddings can be used for semantic retrieval.

## Exercise 3. Semantic Retrieval with Embeddings

Keyword search looks for matching words.

Semantic retrieval compares the meaning of a question with the meaning of documents.

In this exercise, we will use three short ESG-related documents to see how semantic retrieval works.

### 3.1 Prepare a Small Document Collection

Create three short documents representing environmental, social, and governance topics.

These documents will form our small searchable collection.

In [4]:
import numpy as np

documents = {
    "DOC1": (
        "Companies can lower their carbon footprint through "
        "energy efficiency and cleaner production."
    ),
    "DOC2": (
        "Employee well-being, workplace safety, diversity, "
        "and community relations are important social issues."
    ),
    "DOC3": (
        "Independent directors, executive compensation, "
        "shareholder rights, and internal controls are key governance issues."
    )
}

### 3.2 Enter a Search Question

Enter a question that uses different wording from the documents.

The system should still identify the document with the most similar meaning.

In [5]:
query = "How can firms reduce greenhouse gas emissions?"

Notice that the question uses **"greenhouse gas emissions"**, while DOC1 uses **"carbon footprint."**

The wording is different, but the meanings are closely related.

### 3.3 Create Embeddings

Convert the documents and the search question into numerical vectors called embeddings.

Embeddings allow computers to compare texts based on semantic meaning rather than exact wording.

In [6]:
embedding_model = "text-embedding-3-small"

document_response = client.embeddings.create(
    model=embedding_model,
    input=list(documents.values())
)

document_embeddings = [
    item.embedding for item in document_response.data
]

query_response = client.embeddings.create(
    model=embedding_model,
    input=query
)

query_embedding = query_response.data[0].embedding


### 3.4 Inspect the Embedding Vectors

Convert the embeddings into NumPy arrays and inspect their dimensions.

The individual numbers are difficult to interpret directly. Their usefulness comes from comparing the vectors mathematically.

In [ ]:
import pandas as pd


preview_size = 8
embedding_preview = []


for doc_id, embedding in zip(
    documents.keys(),
    document_embeddings
):
    row = {
        "Text ID": doc_id,
        "Dimension": len(embedding)
    }

    for i, value in enumerate(
        embedding[:preview_size],
        start=1
    ):
        row[f"V{i}"] = round(value, 6)

    embedding_preview.append(row)


# Add the query embedding
query_row = {
    "Text ID": "QUERY",
    "Dimension": len(query_embedding)
}

for i, value in enumerate(
    query_embedding[:preview_size],
    start=1
):
    query_row[f"V{i}"] = round(value, 6)

embedding_preview.append(query_row)


preview_df = pd.DataFrame(embedding_preview)
display(preview_df)

### 3.5 Define Cosine Similarity

Cosine similarity measures how similar two vectors are in direction.

A higher score indicates greater semantic similarity between two texts.

In [9]:
def cosine_similarity(vector_a, vector_b):
    """Calculate cosine similarity between two vectors."""

    a = np.array(vector_a)
    b = np.array(vector_b)

    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        return 0.0

    return np.dot(a, b) / denominator


### 3.6 Calculate and Rank Similarity Scores

Compare the search question with each document.

The documents are ranked from the highest to the lowest semantic similarity.

In [10]:
results = []

for doc_id, doc_embedding in zip(
    documents.keys(),
    document_embeddings
):
    score = cosine_similarity(
        query_embedding,
        doc_embedding
    )

    results.append({
        "doc_id": doc_id,
        "text": documents[doc_id],
        "score": score
    })

results = sorted(
    results,
    key=lambda x: x["score"],
    reverse=True
)


### 3.7 Display the Retrieval Results

Display the documents in order of semantic similarity to the search question.

The document with the highest score should be the most relevant to the meaning of the question.

In [ ]:
print(f"Query: {query}\n")

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['doc_id']} "
        f"(Similarity: {result['score']:.4f})"
    )
    print(result["text"])
    print()